# 04 — Grad-CAM Visual Explainability

This notebook generates Grad-CAM heatmaps for the trained EfficientNet-B3 detector, revealing which image regions drive REAL vs FAKE classifications.

**This notebook:**
1. Mounts Google Drive, clones the repo, and installs dependencies
2. Loads the trained checkpoint from notebook 02
3. Samples images from CIFAKE test set (REAL and FAKE)
4. Generates Grad-CAM heatmaps for each sample
5. Displays side-by-side grids: original image vs heatmap overlay
6. Saves all heatmaps to `outputs/heatmaps/` and copies to Google Drive

**Prerequisites:** Run `02_baseline_training.ipynb` first to produce a trained checkpoint.

In [ ]:
# Cell 1 — Mount Google Drive, clone repo
import os
from google.colab import drive

drive.mount('/content/drive')

REPO_DIR = '/content/ai-image-detection'
DRIVE_ROOT = '/content/drive/MyDrive/ai-image-detection'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone https://github.com/krishi-shah/ai-image-detection.git {REPO_DIR}')

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# Cell 2 — Install dependencies
!pip install -q -r requirements.txt
!pip install -q grad-cam

In [ ]:
# Cell 3 — Extract CIFAKE dataset (skips if already extracted)
import glob, zipfile

DATA_DIR = os.path.join(REPO_DIR, 'data', 'raw', 'cifake')
os.makedirs(DATA_DIR, exist_ok=True)

if os.path.exists(os.path.join(DATA_DIR, 'train')):
    print('CIFAKE already extracted — skipping.')
else:
    zip_candidates = glob.glob(os.path.join(DRIVE_ROOT, '*.zip'))
    if not zip_candidates:
        raise FileNotFoundError(f'No zip file found in {DRIVE_ROOT}.')
    zip_path = zip_candidates[0]
    print(f'Extracting {zip_path} ...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(DATA_DIR)
    extracted = os.listdir(DATA_DIR)
    if 'train' not in extracted and len(extracted) == 1:
        DATA_DIR = os.path.join(DATA_DIR, extracted[0])
    print(f'Done. DATA_DIR contents: {os.listdir(DATA_DIR)}')

In [ ]:
# Cell 4 — Imports, paths, device, load checkpoint
import sys
import random
import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt
from torchvision import transforms

sys.path.insert(0, REPO_DIR)

CHECKPOINT_DIR = os.path.join(DRIVE_ROOT, 'checkpoints')
HEATMAP_DIR = os.path.join(REPO_DIR, 'outputs', 'heatmaps')
PLOTS_DIR = os.path.join(REPO_DIR, 'outputs', 'plots')
os.makedirs(HEATMAP_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

from src.model.detector import build_detector, load_checkpoint

model = build_detector(pretrained=False).to(device)
checkpoint_path = os.path.join(CHECKPOINT_DIR, 'best_detector.pth')
epoch_loaded, ckpt_metrics = load_checkpoint(model, checkpoint_path)
model.eval()

print(f'Loaded checkpoint from epoch {epoch_loaded}')
print(f'  val_acc: {ckpt_metrics["val_acc"]:.4f}')

In [ ]:
# Cell 5 — Define transforms and target layer for Grad-CAM
eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

target_layer = model.blocks[-1]
CLASS_NAMES = ['FAKE', 'REAL']
print(f'Target layer: {type(target_layer).__name__}')
print(f'EfficientNet-B3 blocks: {len(model.blocks)}')

In [ ]:
# Cell 6 — Helper functions
def load_image(img_path):
    img = Image.open(img_path).convert('RGB')
    img_resized = img.resize((224, 224), Image.BILINEAR)
    img_np = np.array(img_resized).astype(np.float32) / 255.0
    tensor = eval_transform(img).unsqueeze(0).to(device)
    return tensor, img_np

def get_prediction(model, tensor):
    with torch.no_grad():
        logits = model(tensor)
        probs = torch.softmax(logits, dim=1)
    pred = probs.argmax(dim=1).item()
    conf = probs[0, pred].item()
    return pred, conf

print('Helper functions ready.')

In [ ]:
# Cell 7 — Generate Grad-CAM heatmaps for N samples per class
from src.explainability.gradcam import run_gradcam, save_heatmap

N_SAMPLES = 4
results = {'REAL': [], 'FAKE': []}

for cls in ['REAL', 'FAKE']:
    folder = os.path.join(DATA_DIR, 'test', cls)
    sampled = random.sample(os.listdir(folder), N_SAMPLES)

    for fname in sampled:
        img_path = os.path.join(folder, fname)
        tensor, img_np = load_image(img_path)
        pred_idx, conf = get_prediction(model, tensor)
        pred_label = CLASS_NAMES[pred_idx]
        heatmap = run_gradcam(model, tensor, target_layer)
        save_path = os.path.join(HEATMAP_DIR, f'{cls}_{fname}')
        save_heatmap(heatmap, img_np, save_path)
        results[cls].append({
            'fname': fname,
            'img_np': img_np,
            'heatmap': heatmap,
            'pred_label': pred_label,
            'conf': conf,
            'correct': pred_label == cls,
        })
        status = 'OK' if pred_label == cls else 'WRONG'
        print(f'[{cls}] {fname} -> pred={pred_label} ({conf:.2%}) [{status}]')

print(f'Heatmaps saved to: {HEATMAP_DIR}')

In [ ]:
# Cell 8 — Display Grad-CAM grid for REAL images
from pytorch_grad_cam.utils.image import show_cam_on_image

def plot_gradcam_grid(class_results, true_label, save_path):
    n = len(class_results)
    fig, axes = plt.subplots(2, n, figsize=(4 * n, 8))
    fig.suptitle(f'Grad-CAM — True class: {true_label}', fontsize=14, fontweight='bold')
    for col, item in enumerate(class_results):
        overlay = show_cam_on_image(item['img_np'], item['heatmap'], use_rgb=True)
        color = 'green' if item['correct'] else 'red'
        axes[0, col].imshow(item['img_np'])
        axes[0, col].set_title('Original', fontsize=9)
        axes[0, col].axis('off')
        axes[1, col].imshow(overlay)
        axes[1, col].set_title(f'Pred: {item["pred_label"]} ({item["conf"]:.1%})', fontsize=9, color=color)
        axes[1, col].axis('off')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_path}')

plot_gradcam_grid(results['REAL'], 'REAL', os.path.join(PLOTS_DIR, 'gradcam_real.png'))

In [ ]:
# Cell 9 — Display Grad-CAM grid for FAKE images
plot_gradcam_grid(results['FAKE'], 'FAKE', os.path.join(PLOTS_DIR, 'gradcam_fake.png'))

In [ ]:
# Cell 10 — Combined REAL vs FAKE comparison grid
n = N_SAMPLES
fig, axes = plt.subplots(4, n, figsize=(4 * n, 14))
row_labels = ['REAL — Original', 'REAL — Grad-CAM', 'FAKE — Original', 'FAKE — Grad-CAM']
for row_idx, label in enumerate(row_labels):
    axes[row_idx, 0].set_ylabel(label, fontsize=10)

for col, item in enumerate(results['REAL']):
    overlay = show_cam_on_image(item['img_np'], item['heatmap'], use_rgb=True)
    axes[0, col].imshow(item['img_np'])
    axes[0, col].axis('off')
    axes[1, col].imshow(overlay)
    axes[1, col].set_title(f'Pred: {item["pred_label"]} ({item["conf"]:.1%})', fontsize=8,
                            color='green' if item['correct'] else 'red')
    axes[1, col].axis('off')

for col, item in enumerate(results['FAKE']):
    overlay = show_cam_on_image(item['img_np'], item['heatmap'], use_rgb=True)
    axes[2, col].imshow(item['img_np'])
    axes[2, col].axis('off')
    axes[3, col].imshow(overlay)
    axes[3, col].set_title(f'Pred: {item["pred_label"]} ({item["conf"]:.1%})', fontsize=8,
                            color='green' if item['correct'] else 'red')
    axes[3, col].axis('off')

fig.suptitle('Grad-CAM: REAL vs FAKE — What the model sees', fontsize=13, fontweight='bold')
plt.tight_layout()
comparison_path = os.path.join(PLOTS_DIR, 'gradcam_comparison.png')
plt.savefig(comparison_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {comparison_path}')

In [ ]:
# Cell 11 — Save all outputs to Google Drive
import shutil

drive_outputs = os.path.join(DRIVE_ROOT, 'outputs')
shutil.copytree(
    os.path.join(REPO_DIR, 'outputs'),
    drive_outputs,
    dirs_exist_ok=True
)
print(f'All outputs saved to Drive: {drive_outputs}')
print('Contents:', os.listdir(drive_outputs))